In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import cv2
import pandas as pd
import numpy as np
import os
import glob
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as calculate_ssim
from skimage.metrics import peak_signal_noise_ratio as calculate_psnr
import zlib

class CFG:
    DIV2K_TRAIN_DIR = "/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR/DIV2K_train_HR" 
    DIV2K_VALID_DIR = "/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_valid_HR/DIV2K_valid_HR"
    
    MODEL_SAVE_PATH = "/kaggle/working/best_v6_dynamic_zlib.pth"
    TRAIN_LOG_CSV   = "/kaggle/working/v6_train_history.csv"
    NUMERIC_CSV     = "/kaggle/working/v6_tam_ekran_zipli_metrikler.csv"
    VISUAL_DIR      = "/kaggle/working/v6_juri_gorselleri"
    
    BOTTLENECK_CHANNELS = 3  
    HARD_THRESHOLD = 0.1     
    
    L1_WEIGHT = 1.0           
    PERCEPTUAL_WEIGHT = 0.1   
    SMART_TV_WEIGHT = 0.05    
    ALPHA_DECAY = 5.0  
    
    DYNAMIC_VALVE_PENALTY = 0.05 
    
    EPOCHS = 200
    LR = 2e-4
    ACCUMULATION_STEPS = 16
    BATCH_SIZE = 1 
    PATIENCE = 7
    MIN_DELTA = 0.00001
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CFG.VISUAL_DIR, exist_ok=True)

class DynamicPatchDataset(Dataset):
    def __init__(self, image_paths): self.image_paths = image_paths
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        crop_h, crop_w = np.random.randint(16, min(h//4, 128)) * 4, np.random.randint(16, min(w//4, 128)) * 4
        y, x = np.random.randint(0, h - crop_h), np.random.randint(0, w - crop_w)
        t = torch.from_numpy(img[y:y+crop_h, x:x+crop_w]).permute(2,0,1).float() / 255.0
        return t, t 

class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features[:9].eval()
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg.to(CFG.DEVICE)
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(CFG.DEVICE)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(CFG.DEVICE)
    def forward(self, pred, target):
        return nn.MSELoss()(self.vgg((pred - self.mean) / self.std), self.vgg((target - self.mean) / self.std))

class SmartTVLoss(nn.Module):
    def __init__(self, alpha=5.0):
        super().__init__()
        self.alpha = alpha
    def forward(self, latent, mask):
        diff_h = torch.abs(latent[:, :, 1:, :] - latent[:, :, :-1, :])
        diff_w = torch.abs(latent[:, :, :, 1:] - latent[:, :, :, :-1])
        weight_h = torch.exp(-self.alpha * mask[:, :, :-1, :])
        weight_w = torch.exp(-self.alpha * mask[:, :, :, :-1])
        return (weight_h * diff_h).mean() + (weight_w * diff_w).mean()


class ImageComplexityEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('kx', kx.repeat(3, 1, 1, 1))
        self.register_buffer('ky', ky.repeat(3, 1, 1, 1))
    def forward(self, x):
        gx = F.conv2d(x, self.kx, padding=1, groups=3)
        gy = F.conv2d(x, self.ky, padding=1, groups=3)
        edges = torch.sqrt(gx**2 + gy**2 + 1e-6)
        return edges.mean(dim=[1,2,3]) 


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(True), nn.Conv2d(channels, channels, 3, padding=1))
    def forward(self, x): return F.relu(x + self.conv(x))

class SplitComputingV6Net(nn.Module):
    def __init__(self, b_channels=3):
        super().__init__()
        self.valve = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(True), ResidualBlock(16), nn.Conv2d(16, 1, 3, padding=1), nn.Sigmoid())
        self.encoder = nn.Sequential(nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(True), ResidualBlock(32), ResidualBlock(32), nn.Conv2d(32, b_channels, 3, padding=1), nn.Sigmoid())
        self.decoder = nn.Sequential(nn.Conv2d(b_channels, 16, 3, padding=1), nn.ReLU(True), nn.Conv2d(16, 3, 3, padding=1), nn.Sigmoid())

    def forward(self, x):
        v_map = self.valve(x) 
        x_fused = torch.cat([x, v_map], dim=1) 
        latent = self.encoder(x_fused) 
        

        keep_mask = (v_map >= CFG.HARD_THRESHOLD).float()
        target_mask = v_map * keep_mask 
        mask_ste = target_mask.detach() - v_map.detach() + v_map if self.training else target_mask
        latent_masked = latent * mask_ste 
        
        latent_quantized = torch.round(latent_masked * 255.0) / 255.0
        latent_final_ste = latent_quantized.detach() - latent_masked.detach() + latent_masked if self.training else latent_quantized
        
        out = self.decoder(latent_final_ste)
        return out, v_map, latent_final_ste


print(f"\n" + "="*50)
print(f"🌟 V6 EĞİTİM MOTORU BAŞLATILIYOR (Cihaz: {CFG.DEVICE}) 🌟")
print("="*50)

all_images = glob.glob(os.path.join(CFG.DIV2K_TRAIN_DIR, "*.png"))
train_paths, val_paths = train_test_split(all_images, test_size=0.10, random_state=42)
train_loader = DataLoader(DynamicPatchDataset(train_paths), batch_size=CFG.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(DynamicPatchDataset(val_paths), batch_size=CFG.BATCH_SIZE, shuffle=False)

model = SplitComputingV6Net(b_channels=CFG.BOTTLENECK_CHANNELS).to(CFG.DEVICE)
vgg_loss_fn = VGGLoss() 
smart_tv_loss_fn = SmartTVLoss(alpha=CFG.ALPHA_DECAY)
complexity_estimator = ImageComplexityEstimator().to(CFG.DEVICE) 
l1_loss_fn = nn.L1Loss()

optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

best_val_loss = float('inf')
patience_counter = 0
history = []

for epoch in range(CFG.EPOCHS):
    model.train()
    running_loss = 0.0
    running_vgg_train = 0.0 
    optimizer.zero_grad()
    
    for i, (img_in, img_tgt) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}"):
        img_in, img_tgt = img_in.to(CFG.DEVICE), img_tgt.to(CFG.DEVICE)
        pred, v_map, latent_data = model(img_in) 
        
        loss_l1 = l1_loss_fn(pred, img_tgt)
        loss_vgg = vgg_loss_fn(pred, img_tgt)
        loss_tv = smart_tv_loss_fn(latent_data, v_map) 
        
        with torch.no_grad():
            complexity = complexity_estimator(img_in)
            
        dynamic_weight = CFG.DYNAMIC_VALVE_PENALTY / (complexity + 0.1)
        loss_valve_penalty = (dynamic_weight * v_map.view(img_in.size(0), -1).mean(dim=1)).mean()
        
        total_loss = (CFG.L1_WEIGHT * loss_l1) + (CFG.PERCEPTUAL_WEIGHT * loss_vgg) + \
                     (CFG.SMART_TV_WEIGHT * loss_tv) + loss_valve_penalty
                     
        (total_loss / CFG.ACCUMULATION_STEPS).backward()
        
        if (i + 1) % CFG.ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        running_loss += total_loss.item()
        running_vgg_train += loss_vgg.item() 
        
    avg_train = running_loss / len(train_loader)
    avg_vgg_train = running_vgg_train / len(train_loader)
    
    model.eval()
    running_val = 0.0
    running_vgg_val = 0.0
    with torch.no_grad():
        for img_in, img_tgt in val_loader:
            img_in, img_tgt = img_in.to(CFG.DEVICE), img_tgt.to(CFG.DEVICE)
            pred, v_map, latent_data = model(img_in)
            
            l1 = l1_loss_fn(pred, img_tgt)
            vgg = vgg_loss_fn(pred, img_tgt)
            tv = smart_tv_loss_fn(latent_data, v_map)
            
            complexity = complexity_estimator(img_in)
            dynamic_weight = CFG.DYNAMIC_VALVE_PENALTY / (complexity + 0.1)
            loss_valve_penalty = (dynamic_weight * v_map.view(img_in.size(0), -1).mean(dim=1)).mean()
            
            val_loss = (CFG.L1_WEIGHT * l1) + (CFG.PERCEPTUAL_WEIGHT * vgg) + (CFG.SMART_TV_WEIGHT * tv) + loss_valve_penalty
            
            running_val += val_loss.item()
            running_vgg_val += vgg.item()
            
    avg_val = running_val / len(val_loader)
    avg_vgg_val = running_vgg_val / len(val_loader)
    scheduler.step(avg_val)
    
    print(f"Epoch {epoch+1} | Train: {avg_train:.5f} | Val: {avg_val:.5f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    history.append({
        "Epoch": epoch+1, 
        "Train_Loss": avg_train, "Val_Loss": avg_val,
        "VGG_Train_Loss": avg_vgg_train, "VGG_Val_Loss": avg_vgg_val
    })
    pd.DataFrame(history).to_csv(CFG.TRAIN_LOG_CSV, index=False)
    
    if (best_val_loss - avg_val) > CFG.MIN_DELTA:
        best_val_loss = avg_val
        patience_counter = 0
        torch.save(model.state_dict(), CFG.MODEL_SAVE_PATH)
    else:
        patience_counter += 1
        if patience_counter >= CFG.PATIENCE:
            break


print(f"\n" + "="*50)
print("="*50)

try:
    test_model = SplitComputingV6Net(b_channels=CFG.BOTTLENECK_CHANNELS).to(CFG.DEVICE)
    test_model.load_state_dict(torch.load(CFG.MODEL_SAVE_PATH, map_location=CFG.DEVICE, weights_only=True))
    test_model.eval()
except Exception as e:
    print("", e)

valid_images = glob.glob(os.path.join(CFG.DIV2K_VALID_DIR, "*.png"))
if not valid_images: valid_images = glob.glob(os.path.join(CFG.DIV2K_TRAIN_DIR, "*.png"))
results = []

for img_path in tqdm(valid_images, desc="V6 Zipli Test (Adil Kıyaslama)"):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    file_name = os.path.basename(img_path)
    
    h, w, _ = img.shape
    h_trim, w_trim = (h // 4) * 4, (w // 4) * 4
    full_frame = img[:h_trim, :w_trim]
    t_in = torch.from_numpy(full_frame).permute(2,0,1).float().unsqueeze(0).to(CFG.DEVICE) / 255.0
    
    original_uint8 = full_frame.astype(np.uint8)
    original_zipped_bytes = zlib.compress(original_uint8.tobytes(), level=9)
    original_zipped_kb = len(original_zipped_bytes) / 1024.0
    
    with torch.no_grad():
        pred, v_map, t_data = test_model(t_in)
        out_np = pred.squeeze(0).permute(1,2,0).cpu().numpy()
        full_frame_np = full_frame / 255.0
        v_map_np = v_map.squeeze().cpu().numpy()
        
        vgg_score = vgg_loss_fn(pred, t_in).item()
        
        t_data_uint8 = (t_data * 255.0).byte().cpu().numpy() 
        raw_bytes = t_data_uint8.tobytes()
        model_zipped_bytes = zlib.compress(raw_bytes, level=9)
        model_zipped_kb = len(model_zipped_bytes) / 1024.0

    raw_uncompressed_kb = (h_trim * w_trim * 3) / 1024.0 
    psnr = calculate_psnr(full_frame_np, out_np, data_range=1.0)
    ssim = calculate_ssim(full_frame_np, out_np, data_range=1.0, channel_axis=2)
    l1_loss = np.mean(np.abs(full_frame_np - out_np))
    zero_ratio = np.mean(v_map_np < CFG.HARD_THRESHOLD)
    
    results.append({
        "Resim": file_name, 
        "PSNR (dB)": round(psnr, 2), "SSIM": round(ssim, 4), 
        "L1_Loss": round(l1_loss, 4), "VGG_Loss": round(vgg_score, 4),
        "Sifirlanan_Alan (%)": round(zero_ratio * 100, 2),
        "Raw_Uncompressed_Size (KB)": round(raw_uncompressed_kb, 2),
        "Original_ZIPPED_KB": round(original_zipped_kb, 2), 
        "Model_ZIPPED_KB": round(model_zipped_kb, 2),       
        "Bilesik_Skor": round(l1_loss + (model_zipped_kb/10000), 5),
        "Image_Path": img_path, "h": h_trim, "w": w_trim
    })

df_all = pd.DataFrame(results)
df_all.drop(columns=['Image_Path', 'h', 'w']).to_csv(CFG.NUMERIC_CSV, index=False)

print(f"\n🎨 Görseller Çiziliyor... Klasör: '{CFG.VISUAL_DIR}'")
df_sorted = df_all.sort_values(by="Bilesik_Skor", ascending=True).reset_index(drop=True)
n = len(df_sorted)
selected_indices = [0, n // 6, 2 * n // 6, n // 2, 4 * n // 6, 5 * n // 6, n - 1]
labels = ["1_Kusursuz", "2_Cok_Iyi", "3_Iyi", "4_Ortalama", "5_Ortalama_Alti", "6_Kotu", "7_En_Basarisiz"]

for i, rank_idx in enumerate(selected_indices):
    row = df_sorted.iloc[rank_idx]
    label_name = labels[i]
    
    img = cv2.cvtColor(cv2.imread(row['Image_Path']), cv2.COLOR_BGR2RGB)
    h_trim, w_trim = row['h'], row['w']
    full_frame = img[:h_trim, :w_trim]
    t_in = torch.from_numpy(full_frame).permute(2,0,1).float().unsqueeze(0).to(CFG.DEVICE) / 255.0
    
    with torch.no_grad():
        pred, v_map, t_data = test_model(t_in)
        out_np = pred.squeeze(0).permute(1,2,0).cpu().numpy()
        v_map_np = v_map.squeeze().cpu().numpy()
        bottle_np = (t_data.squeeze(0).mean(dim=0).cpu().numpy() * 255).astype(np.uint8)
        full_frame_np = full_frame / 255.0

    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    title_str = f"[V6 DİNAMİK BÜTÇE | {label_name.replace('_', ' ')}] PSNR: {row['PSNR (dB)']} dB\n" \
                f"Orijinal ZLIB: {row['Original_ZIPPED_KB']} KB -> Model ZLIB: {row['Model_ZIPPED_KB']} KB"
    fig.suptitle(title_str, fontsize=18, fontweight='bold', color='purple')
    
    axes[0, 0].imshow(full_frame); axes[0, 0].set_title("1. Orijinal 2K Giriş")
    im_valve = axes[0, 1].imshow(v_map_np, cmap='inferno', vmin=0, vmax=1)
    axes[0, 1].set_title(f"2. Önem Maskesi (%{row['Sifirlanan_Alan (%)']} Sıfırlandı)")
    fig.colorbar(im_valve, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    im_bottle = axes[0, 2].imshow(bottle_np, cmap='viridis')
    axes[0, 2].set_title(f"3. 8-Bit Tamsayı Darboğaz ({row['Model_ZIPPED_KB']} KB)")
    fig.colorbar(im_bottle, ax=axes[0, 2], fraction=0.046, pad=0.04)
    
    axes[1, 0].imshow(out_np); axes[1, 0].set_title("4. İstemci Çıktısı (Onarılmış)")
    im_err = axes[1, 1].imshow(np.abs(full_frame_np - out_np).mean(axis=2), cmap='hot', vmin=0, vmax=0.1)
    axes[1, 1].set_title("5. Hata (Kayıp) Haritası")
    fig.colorbar(im_err, ax=axes[1, 1], fraction=0.046, pad=0.04)
    
    overlay = full_frame_np * np.expand_dims(v_map_np, axis=-1)
    axes[1, 2].imshow(overlay); axes[1, 2].set_title("6. Orijinal Üz. Maske")
    
    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    plt.savefig(os.path.join(CFG.VISUAL_DIR, f"{label_name}.png"), dpi=200)
    plt.close()


print("\n" + "="*75)
print("🏆 MİMARİ V6 (DİNAMİK BÜTÇE & ADİL ZLIB KIYASLAMASI) FİNAL SONUÇLAR 🏆")
print("="*75)
print(f"Test Edilen Resim           : {len(df_all)}")
print(f"Ortalama PSNR (Kalite)      : {df_all['PSNR (dB)'].mean():.2f} dB")
print(f"Ortalama VGG (Doku) Skoru   : {df_all['VGG_Loss'].mean():.4f}")
print(f"Dinamik Sıfırlanan Alan     : %{df_all['Sifirlanan_Alan (%)'].mean():.2f}")
print("-" * 75)
print(f"1. Ham Boyut (ZIP'siz)      : {df_all['Raw_Uncompressed_Size (KB)'].mean():.2f} KB")
print(f"2. ORİJİNAL RESİM ZLIB      : {df_all['Original_ZIPPED_KB'].mean():.2f} KB (Adil Baseline)")
print(f"3. BİZİM MODEL ZLIB         : {df_all['Model_ZIPPED_KB'].mean():.2f} KB (Yapay Zeka)")
print(f"🚀 YZ'NİN ZLIB'E KATKISI    : %{100 * (1 - df_all['Model_ZIPPED_KB'].mean() / df_all['Original_ZIPPED_KB'].mean()):.2f} Ekstra Tasarruf!")
print("="*75)